# 04 — Seaborn: vizualizacija podataka o zaradama (Salaries.csv)

**Nastavak vežbe iz Pandas-a.** Koristimo isti skup podataka kako bismo od tabelarne analize prešli na vizuelno istraživanje.

**Ciljevi:** da razlikujemo raspodelu, poređenje kategorija, trend, povezanost i ekstremne vrednosti; da izaberemo odgovarajući grafik; i da kritički tumačimo rezultat.

> Podaci obuhvataju evidencije zarada u San Francisku tokom 2011–2014. Jedan red predstavlja **zapis**, ne nužno jednu jedinstvenu osobu. Valuta u grafikonima je USD. Grafikoni prikazuju deskriptivne odnose, ne uzročno-posledične veze.

## 0. Priprema okruženja i učitavanje podataka

Potrebni paketi: `pandas`, `numpy`, `matplotlib`, `seaborn`. Ako nedostaju, u terminalu pokrenite `python -m pip install pandas numpy matplotlib seaborn`.

Datoteku `Salaries.csv` postavite u isti direktorijum kao ovaj notebook.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

sns.set_theme(style="whitegrid", context="notebook")
DATA_PATH = Path("Salaries.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Postavite Salaries.csv pored notebook-a.")
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Broj zapisa: {len(df):,}; broj kolona: {df.shape[1]}")
display(df.head())

### Proveravamo strukturu i nedostajuće vrednosti

Za razliku od prethodne vežbe, nećemo automatski popunjavati sve nedostajuće vrednosti: time bismo mogli da iskrivimo grafikone. Za pojedine analize koristimo samo redove sa odgovarajućim vrednostima.

In [ ]:
display(df.isna().sum().rename("Nedostajuće vrednosti").to_frame())
display(df[["BasePay", "OvertimePay", "OtherPay", "Benefits", "TotalPay", "TotalPayBenefits"]].describe().round(2))

### Pomoćne promenljive i ograničenja

- `Benefits` nedostaje kod velikog broja zapisa, posebno za 2011. godinu; to **nije isto što i nula**.
- `TotalPayBenefits` je zbir ukupne zarade i dostupnih beneficija, ali poređenje godina zahteva oprez zbog različite pokrivenosti beneficijama.
- U podacima postoje i nulte i negativne obračunske vrednosti. Za raspodele *pozitivnih zarada* jasno ćemo filtrirati pozitivne iznose, a originalne podatke sačuvati.
- Nazivi poslova mogu se razlikovati samo po velikim i malim slovima; koristićemo normalizovan naziv kao pomoćnu kategoriju.

In [ ]:
rad = df.copy()
rad["JobTitle_clean"] = rad["JobTitle"].fillna("NEPOZNATO").str.strip().str.casefold().str.title()
rad["OvertimeShare"] = np.where(rad["TotalPay"] > 0, rad["OvertimePay"] / rad["TotalPay"], np.nan)
rad["BenefitsKnown"] = rad["Benefits"].notna()
fmt_usd = FuncFormatter(lambda x, _: f"${x:,.0f}")
fmt_k = FuncFormatter(lambda x, _: f"{x/1000:,.0f}k")
print("Udeo zapisa bez poznatih beneficija po godini:")
display(rad.groupby("Year")["BenefitsKnown"].agg(["count", "mean"]).rename(columns={"count":"Broj zapisa", "mean":"Udeo sa poznatim beneficijama"}).round(3))

## 1. Koliko zapisa imamo po godini? — `countplot`

**Pitanje:** Da li se broj evidentiranih zapisa menja od 2011. do 2014?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=rad, x="Year", color="steelblue", ax=ax)
ax.set(title="Broj zapisa o zaradama po godini", xlabel="Godina", ylabel="Broj zapisa")
plt.tight_layout(); plt.show()

**Za diskusiju:** Da li je opravdano reći da je porastao broj zaposlenih? Ne nužno — prikazan je broj **zapisa** u skupu podataka.

## 2. Kako su raspoređene osnovne zarade? — `histplot`

Histogram prikazuje koliko se vrednosti nalazi u zadatim intervalima (*bins*). Koristimo pozitivne vrednosti `BasePay`.

In [ ]:
osnovne = rad.loc[rad["BasePay"] > 0, "BasePay"]
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(osnovne, bins=45, color="teal", ax=ax)
ax.xaxis.set_major_formatter(fmt_k)
ax.set(title="Raspodela pozitivnih osnovnih zarada", xlabel="Osnovna zarada (USD)", ylabel="Broj zapisa")
plt.tight_layout(); plt.show()

**Pitanja:** Gde je najviše zapisa? Da li je raspodela simetrična? Kako biste objasnili razliku između proseka i medijane?

In [ ]:
print(f"Prosek pozitivnih osnovnih zarada: ${osnovne.mean():,.2f}")
print(f"Medijana pozitivnih osnovnih zarada: ${osnovne.median():,.2f}")

## 3. Histogram i KDE — isti podaci, dva pogleda

`kdeplot` prikazuje zaglađenu procenu gustine, dok histogram pokazuje broj zapisa po intervalu.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(osnovne, bins=45, stat="density", color="steelblue", alpha=0.35, ax=ax, label="Histogram")
sns.kdeplot(x=osnovne, ax=ax, color="darkred", label="KDE")
ax.xaxis.set_major_formatter(fmt_k)
ax.set(title="Histogram i KDE osnovnih zarada", xlabel="Osnovna zarada (USD)", ylabel="Gustina")
ax.legend(); plt.tight_layout(); plt.show()

**Za diskusiju:** KDE je *procena* raspodele — njen oblik zavisi od izabranog nivoa zaglađivanja.

## 4. Raspodela zarada po godinama — `boxplot`

Boxplot prikazuje medijanu, kvartile i moguće ekstremne vrednosti. Koristimo pozitivne osnovne zarade, bez imputacije.

In [ ]:
podaci = rad.loc[rad["BasePay"] > 0]
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=podaci, x="Year", y="BasePay", color="lightblue", showfliers=False, ax=ax)
ax.yaxis.set_major_formatter(fmt_usd)
ax.set(title="Osnovne zarade po godinama (bez prikaza autlajera)", xlabel="Godina", ylabel="Osnovna zarada (USD)")
plt.tight_layout(); plt.show()

**Za diskusiju:** Da li se medijana menja? Zašto smo sakrili prikaz autlajera (`showfliers=False`), a nismo ih izbacili iz podataka?

## 5. Prosečna i medijalna osnovna zarada po godinama — `lineplot`

Grupisanje sa `groupby` iz Pandas vežbe sada pretvaramo u grafik. **Važno:** ovo su vrednosti po zapisu, ne praćenje plata istih ljudi kroz godine; iznosi nisu prilagođeni inflaciji.

In [ ]:
godisnje = rad.groupby("Year", as_index=False).agg(prosek=("BasePay", "mean"), medijana=("BasePay", "median"))
godisnje_dugo = godisnje.melt(id_vars="Year", value_vars=["prosek", "medijana"], var_name="Mera", value_name="USD")
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=godisnje_dugo, x="Year", y="USD", hue="Mera", marker="o", ax=ax)
ax.set_xticks(sorted(rad["Year"].unique()))
ax.yaxis.set_major_formatter(fmt_usd)
ax.set(title="Prosečna i medijalna osnovna zarada po godinama", xlabel="Godina", ylabel="USD")
plt.tight_layout(); plt.show()
display(godisnje.round(2))

**Pitanje:** Šta bi se promenilo kada bismo koristili prosečnu umesto medijalne zarade?

## 6. Pet najčešćih poslovnih pozicija — `barplot`

S obzirom na različitu upotrebu velikih i malih slova, prvo grupišemo normalizovane nazive poslova. Zatim prikazujemo **broj zapisa**, a ne broj različitih zaposlenih.

In [ ]:
top_poslovi = rad["JobTitle_clean"].value_counts().head(5).rename_axis("Pozicija").reset_index(name="Broj zapisa")
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=top_poslovi, y="Pozicija", x="Broj zapisa", color="slateblue", ax=ax)
ax.set(title="Pet najčešćih pozicija u evidenciji", xlabel="Broj zapisa", ylabel="")
plt.tight_layout(); plt.show()
display(top_poslovi)

**Pitanje:** Zašto bi rezultat bio drugačiji ako bismo koristili originalnu kolonu `JobTitle` bez normalizacije?

## 7. Koliko se osnovne zarade razlikuju između čestih poslova? — `boxplot`

Za pet najčešćih naziva poslova posmatramo raspodelu **pozitivnih** osnovnih zarada.

In [ ]:
pozicije = top_poslovi["Pozicija"].tolist()
za_poslove = rad.loc[(rad["JobTitle_clean"].isin(pozicije)) & (rad["BasePay"] > 0)]
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=za_poslove, y="JobTitle_clean", x="BasePay", order=pozicije, color="wheat", showfliers=False, ax=ax)
ax.xaxis.set_major_formatter(fmt_k)
ax.set(title="Raspodela osnovnih zarada — pet najčešćih pozicija", xlabel="Osnovna zarada (USD)", ylabel="")
plt.tight_layout(); plt.show()

**Pitanje:** Koja pozicija ima najvišu medijanu? Da li boxplot govori nešto o broju zapisa u svakoj kategoriji?

## 8. Osnovna zarada i ukupna zarada — `scatterplot`

Istražujemo odnos između `BasePay` i `TotalPay`. Radi preglednosti prikazujemo **slučajan uzorak od najviše 5.000 zapisa** sa obe poznate, pozitivne vrednosti. Uzorak je reproduktibilan (`random_state=42`).

In [ ]:
scatter_data = rad.loc[(rad["BasePay"] > 0) & (rad["TotalPay"] > 0), ["BasePay", "TotalPay", "Year"]]
scatter_uzorak = scatter_data.sample(n=min(5000, len(scatter_data)), random_state=42)
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=scatter_uzorak, x="BasePay", y="TotalPay", hue="Year", palette="viridis", alpha=0.35, s=20, ax=ax)
maximum = max(scatter_uzorak["BasePay"].max(), scatter_uzorak["TotalPay"].max())
ax.plot([0, maximum], [0, maximum], linestyle="--", color="black", linewidth=1, label="Jednake vrednosti")
ax.xaxis.set_major_formatter(fmt_k); ax.yaxis.set_major_formatter(fmt_k)
ax.set(title="Osnovna i ukupna zarada — slučajan uzorak", xlabel="BasePay (USD)", ylabel="TotalPay (USD)")
plt.tight_layout(); plt.show()

**Pitanja:** Da li tačke leže iznad linije jednakosti? Šta objašnjava razliku između osnovne i ukupne zarade? Da li korelacija automatski znači uzročnost?

## 9. Gde je izražen prekovremeni rad? — `boxplot` i `barplot`

Upoređujemo **pozitivne** prekovremene isplate po godinama i udeo zapisa sa prekovremenom isplatom. Ovo ne predstavlja broj odrađenih sati.

In [ ]:
prekovremeni = rad.loc[rad["OvertimePay"] > 0]
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=prekovremeni, x="Year", y="OvertimePay", color="salmon", showfliers=False, ax=ax)
ax.yaxis.set_major_formatter(fmt_usd)
ax.set(title="Prekovremene isplate među zapisima sa pozitivnom isplatom", xlabel="Godina", ylabel="OvertimePay (USD)")
plt.tight_layout(); plt.show()

In [ ]:
udeo_preko = rad.groupby("Year")["OvertimePay"].apply(lambda s: (s > 0).mean() * 100).reset_index(name="Udeo (%)")
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=udeo_preko, x="Year", y="Udeo (%)", color="coral", ax=ax)
ax.set(title="Udeo zapisa sa pozitivnom prekovremenom isplatom", xlabel="Godina", ylabel="Udeo svih zapisa (%)", ylim=(0, 100))
plt.tight_layout(); plt.show()
display(udeo_preko.round(2))

**Pitanje:** Zašto nije dovoljno posmatrati samo prosečan `OvertimePay` da bismo razumeli koliko je prekovremeni rad rasprostranjen?

## 10. Beneficije — važnost nedostajućih podataka

Pre poređenja iznosa beneficija proveravamo koji udeo zapisa u svakoj godini zaista ima unetu vrednost `Benefits`. **Nepoznatu beneficiju ne smemo automatski tumačiti kao nulu.**

In [ ]:
pokrivenost = rad.groupby("Year")["BenefitsKnown"].mean().mul(100).reset_index(name="Poznato (%)")
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=pokrivenost, x="Year", y="Poznato (%)", color="mediumseagreen", ax=ax)
ax.set(title="Pokrivenost podataka o beneficijama", xlabel="Godina", ylabel="Zapisi sa poznatim Benefits (%)", ylim=(0, 105))
plt.tight_layout(); plt.show()
display(pokrivenost.round(1))

**Za diskusiju:** Zašto bi grafik prosečnih beneficija za 2011–2014 mogao da zavara? Da li je poređenje `TotalPayBenefits` između 2011. i narednih godina bez dodatnih ograničenja opravdano?

## 11. Šta čini ukupnu zaradu? — struktura komponenti po godini

Prikazujemo prosečne iznose `BasePay`, `OvertimePay` i `OtherPay` po zapisu, i to samo za zapise gde su sve tri komponente poznate. **Ne uključujemo `Benefits`** jer nisu jednako dostupne kroz godine.

In [ ]:
komponente = ["BasePay", "OvertimePay", "OtherPay"]
valid = rad.dropna(subset=komponente)
sastav = valid.groupby("Year")[komponente].mean()
ax = sastav.plot(kind="bar", stacked=True, figsize=(10, 5), color=["#4C78A8", "#F58518", "#54A24B"])
ax.yaxis.set_major_formatter(fmt_usd)
ax.set(title="Prosečne komponente ukupne zarade po zapisu", xlabel="Godina", ylabel="Prosečan iznos (USD)")
ax.legend(title="Komponenta")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
display(sastav.round(2))

**Pitanje:** Koja komponenta najviše doprinosi prosečnoj ukupnoj zaradi? Da li na osnovu ovog grafikona možemo zaključiti nešto o pojedinačnom zaposlenom?

## 12. Korelacije numeričkih atributa — `heatmap`

Korelacija ukazuje na jačinu linearne povezanosti. Izostavljamo `Id` i `Year` kao identifikator i vremensku oznaku. **Oprez:** `TotalPay` i `TotalPayBenefits` su izvedeni iz drugih kolona pa njihove korelacije nisu nezavisni nalazi.

In [ ]:
brojcane = ["BasePay", "OvertimePay", "OtherPay", "Benefits", "TotalPay", "TotalPayBenefits"]
korelacije = rad[brojcane].corr()
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(korelacije, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set(title="Korelacije komponenti zarada (dostupni parovi vrednosti)")
plt.tight_layout(); plt.show()

**Pitanja:** Koji par promenljivih je najjače povezan? Zašto je korelacija `BasePay`–`TotalPay` očekivana? Zašto korelacija ne dokazuje uzročnost?

## 13. Ekstremne vrednosti — deset najvećih ukupnih zarada

Posmatramo **10 zapisa** sa najvećim `TotalPay`, ne tvrdimo da se radi o deset različitih ljudi. Obavezno navodimo godinu.

In [ ]:
top10 = rad.nlargest(10, "TotalPay").copy()
top10["Oznaka"] = top10["EmployeeName"].astype(str) + " (" + top10["Year"].astype(str) + ")"
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=top10, x="TotalPay", y="Oznaka", color="indianred", ax=ax)
ax.xaxis.set_major_formatter(fmt_usd)
ax.set(title="Deset zapisa sa najvećom ukupnom zaradom", xlabel="TotalPay (USD)", ylabel="")
plt.tight_layout(); plt.show()
display(top10[["EmployeeName", "JobTitle", "Year", "BasePay", "OvertimePay", "OtherPay", "TotalPay"]])

**Za diskusiju:** Da li su najviši iznosi posledica samo velike osnovne zarade? Kako bi ove ekstremne vrednosti mogle uticati na prosek?

## 14. Samostalni rad — interpretacija i dodatni grafikoni

**Zadatak A.** Kreirajte `violinplot` osnovnih zarada po godinama (koristite samo pozitivne iznose). Uporedite ga sa boxplot-om iz sekcije 4.

**Zadatak B.** Pronađite pet pozicija sa najvećim *medijalnim* `TotalPay` među pozicijama koje imaju najmanje **100 zapisa**. Rezultate predstavite horizontalnim barplot-om. Objasnite zašto je prag od 100 zapisa koristan.

**Zadatak C.** Napravite histogram pozitivnih prekovremenih isplata i istražite kako se menja izgled grafikona kada promenite `bins`.

**Zadatak D.** Sastavite tri zaključka zasnovana na grafikonima i navedite najmanje dva ograničenja skupa podataka.

**Bonus:** Da li bi bilo opravdano zaključiti da su zarade realno rasle u periodu 2011–2014? Šta bi još trebalo da znamo?

In [ ]:
# Prostor za zadatak A

In [ ]:
# Prostor za zadatak B

In [ ]:
# Prostor za zadatak C

---
### Nastavna napomena

Primeri koriste `Seaborn` za distribucije, poređenja kategorija, odnose i korelacije, uz `Matplotlib` za formatiranje i jedan složeni (stacked) bar-grafik. Pokretati ćelije redom, počevši od učitavanja podataka.

**Izvor podataka:** priloženi `Salaries.csv`. Rezultati su deskriptivni i ograničeni na njegov sadržaj.